# Expanded trading-research evidence

This notebook is a read-only, executable showcase of repository components and completed immutable evidence. It does not duplicate training logic or select models. Set `TRADING_REPORT_DIR` to either a Slice 1 report containing `provenance.json` or a completed expanded report under `reports/expanded_closeout/<run_id>` containing `research_evidence_manifest.json`. The currently checked-in reports are Slice 1 / diagnostic evidence; expanded-study tables appear only when a generated report contains them. Their absence is not evidence of a zero result or a failed model. The expanded close-out study remains exploratory until a genuinely untouched confirmation period exists.

## End-to-end implementation map

- [Configuration and authoritative runner](../src/trading_pipeline/run.py)
- [Market and SEC extraction](../src/trading_pipeline/data/pipeline.py)
- [Security master](../src/trading_pipeline/data/security_master.py)
- [Point-in-time feature construction](../src/trading_pipeline/features/build.py)
- [Model contracts and registry](../src/trading_pipeline/experiments/contracts.py)
- [Temporal optimisation](../src/trading_pipeline/optimisation/splits.py)
- [Supervised model families](../src/trading_pipeline/modelling/supervised_models)
- [RL policies and environment](../src/trading_pipeline/rl)
- [Portfolio execution and costs](../src/trading_pipeline/portfolio/backtest.py)
- [Risk-return frontier evidence](../src/trading_pipeline/portfolio/frontier.py)
- [Generated reporting](../src/trading_pipeline/reporting/generate.py)
- [Read-only dashboard](../src/trading_pipeline/dashboard/app.py)

## Problem, methodology and core concepts

The research question is whether models using daily market information and point-in-time SEC fundamentals can improve five-session cross-sectional stock ranking, and whether that improvement survives portfolio construction and transaction costs. The workflow deliberately separates prediction, portfolio allocation and economic evaluation so that a forecasting improvement is not mistaken for a profitable strategy.

**Point-in-time information.** A historical decision may use only information public at that time. SEC facts become eligible only when the market session is strictly after the filing date. Rolling market features use observations through signal date T, and the implemented backtest enters at the next session's close, T+1. The five-session close-to-close target and tradable T+1 holding window are not identical.

**Temporal validation.** Training fits parameters; chronological validation chooses among declared configurations; outer walk-forward evaluation estimates the complete procedure; and the completed Slice 1 final holdout is descriptive after its selection lock. It must not be used to choose, tune, or reject features, models, portfolio engines, or parameters. Purging removes samples whose five-session label overlaps a later split, while embargo withholds additional boundary sessions. A new selection study needs a new holdout/data vintage and documented protocol.

**Forecast metrics.** MAE and RMSE are errors, so lower is better. Daily information coefficient (IC) is the Spearman rank correlation between predicted and realised returns across stocks, so higher is better. IC is especially relevant because the portfolio selects the highest-ranked securities, but useful ranking does not guarantee useful after-cost returns.

**Portfolio metrics.** Annualised return measures compounded growth on a 252-session convention; higher is generally preferable subject to risk and uncertainty. Annualised volatility measures return variability; lower is less variable, not necessarily better in isolation. The Sharpe ratio divides annualised mean return by annualised volatility under the stated risk-free-rate convention; higher means more return per unit of variability, but it is sample-dependent and does not capture every tail risk. Maximum drawdown is the worst observed peak-to-trough decline; a smaller decline is generally preferable. Turnover measures traded value relative to equity and connects model instability to transaction costs.

**Benchmark comparison.** Beta measures co-movement with SPY. Tracking error measures variability of active return, and the information ratio compares active return with that tracking error. These help distinguish genuine benchmark-relative behaviour from ordinary market exposure.

**Risk views.** A model-conditioned efficient frontier uses a supervised model's expected returns together with a common covariance estimate and constraints. It is not universal. A realised risk-return curve instead connects the conservative, balanced and aggressive backtest outcomes for a model. DQN and PPO belong in the realised comparison because their sleeve-selector policies do not emit a stock-level expected-return vector.

The notebook displays only values loaded from the selected report's generated Parquet tables. The report loader verifies declared output hashes where available. It does not rebuild metrics or infer absent expanded-study evidence. See the [full glossary and methodological cautions](../docs/GLOSSARY.md) for definitions used throughout the report.

In [ ]:
from pathlib import Path
import os
import polars as pl
try:
    from IPython.display import display, Markdown
except ImportError:  # Script-mode validation without a notebook runtime
    def display(value): return value
    def Markdown(value): return value
from trading_pipeline.dashboard import discover_expanded_reports, load_expanded_report, load_report

def repository_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository checkout.')

root = repository_root()
configured = os.environ.get('TRADING_REPORT_DIR')
if configured:
    report_dir = Path(configured).expanduser().resolve()
else:
    expanded_candidates = discover_expanded_reports(root)
    if expanded_candidates:
        report_dir = expanded_candidates[0]['root']
    else:
        candidates = sorted((root / 'reports').glob('*/v*'), key=lambda path: path.stat().st_mtime)
        if not candidates:
            raise FileNotFoundError('Generate a report or set TRADING_REPORT_DIR.')
        report_dir = candidates[-1]
expanded_report = (report_dir / 'research_evidence_manifest.json').is_file()
loaded = load_expanded_report(report_dir) if expanded_report else load_report(report_dir)
provenance, tables = loaded['provenance'], loaded['tables']
report_label = report_dir.relative_to(root) if report_dir.is_relative_to(root) else report_dir
display(Markdown(f"**Source run:** `{provenance.get('source_run_id')}`  \\n**Report:** `{report_label}`  \\n**Research status:** `{provenance.get('claim_status') if expanded_report else provenance.get('source_run_research_status', 'not recorded')}`"))

## Data extraction, feature store and temporal splits

The report provenance identifies its source run and generated inputs. Tables below are generated report products; they are not recomputed in this notebook. Optional expanded-study tables display only when present. The linked implementation map points to repository code, while the selected run/report records archived execution evidence.

In [ ]:
for name in ('pipeline_stage_summary', 'universe_summary', 'split_profile', 'feature_summary'):
    if name in tables:
        display(Markdown(f'### {name.replace("_", " " ).title()}'))
        display(tables[name])

## Architecture selection, HPO and training

When trial summaries are present, compare candidates using their declared objective and metric direction (for example, lower RMSE or higher IC where those are declared). A leaderboard is descriptive evidence of the declared selection protocol; do not rank models using the final holdout. Missing trial tables mean this report does not provide that evidence.

In [ ]:
for name in ('architecture_trial_summary', 'hpo_trial_summary', 'training_summary', 'training_trace'):
    if name in tables:
        frame = tables[name]
        display(Markdown(f'### {name.replace("_", " " ).title()}'))
        display(frame)

## Risk scenarios and frontiers

Model-conditioned efficient frontiers apply only to supervised predictors that emit security-level expected returns. Realised risk-return curves are the common cross-family comparison and include DQN/PPO.

In [ ]:
import matplotlib.pyplot as plt

frontier = tables.get('model_conditioned_frontier_points')
if frontier is not None and not frontier.is_empty():
    latest = frontier['rebalance_date'].max()
    data = frontier.filter((pl.col('rebalance_date') == latest) & (pl.col('status') == 'complete'))
    fig, ax = plt.subplots()
    for model in data['model_id'].unique().sort().to_list():
        series = data.filter(pl.col('model_id') == model).sort('expected_volatility')
        ax.plot(series['expected_volatility'], series['expected_return'], marker='o', label=model)
    ax.set(xlabel='Expected annualised volatility', ylabel='Expected annualised return', title=f'Model-conditioned frontiers: {latest}')
    ax.legend(); plt.show()
else:
    display(Markdown('*No completed model-conditioned frontier evidence is available.*'))

realised = tables.get('realised_risk_return_curve')
if realised is not None and not realised.is_empty():
    fig, ax = plt.subplots()
    groups = realised.select(['model_id', 'seed']).unique().iter_rows() if 'seed' in realised.columns else ((model, None) for model in realised['model_id'].unique().sort().to_list())
    for model, seed in groups:
        series = realised.filter((pl.col('model_id') == model) & (pl.col('seed') == seed) if seed is not None else pl.col('model_id') == model).sort('risk_order')
        label = f'{model} · seed {seed}' if seed is not None else model
        style = '-' if series['risk_control_monotonic'].all() else 'None'
        ax.plot(series['annualised_volatility'], series['annualised_return'], linestyle=style, marker='o', label=label)
        for row in series.iter_rows(named=True):
            ax.annotate(row['risk_scenario'], (row['annualised_volatility'], row['annualised_return']))
    ax.set(xlabel='Realised annualised volatility', ylabel='Realised annualised return', title='Realised model risk-return curves')
    ax.legend(); plt.show()
    display(realised)
else:
    display(Markdown('*No completed three-scenario realised risk-return evidence is available.*'))

## Final test bench and market comparison

In [ ]:
has_final_bench = 'final_testbench_metrics' in tables
comparison = tables['final_testbench_metrics'] if has_final_bench else tables['experiment_comparison']
display(comparison)
equity_source = tables['final_testbench_equity_curve'] if has_final_bench else tables['equity_drawdown_series']
equity = equity_source.filter(pl.col('split').is_in(['test', 'descriptive_holdout']))
fig, ax = plt.subplots()
groups = equity.select(['experiment_id', 'seed', 'risk_scenario']).unique().iter_rows() if 'seed' in equity.columns else ((experiment, None, None) for experiment in equity['experiment_id'].unique().sort().to_list())
for experiment, seed, scenario in groups:
    cell = ((pl.col('experiment_id') == experiment) & (pl.col('seed') == seed) & (pl.col('risk_scenario') == scenario)) if seed is not None else (pl.col('experiment_id') == experiment)
    series = equity.filter(cell).sort('session_date')
    label = f'{experiment} · {scenario} · seed {seed}' if seed is not None else experiment
    ax.plot(series['session_date'], series['equity'], label=label)
ax.set(xlabel='Session', ylabel='Equity (initial = 1)', title='Final test-bench performance against the market')
ax.legend(ncol=2); plt.show()

## Individual-security contribution

In [ ]:
security = tables['pipeline_security_summary'] if 'pipeline_security_summary' in tables else tables['security_contribution_summary']
ticker = os.environ.get('TRADING_TICKER')
if ticker and 'ticker' in security.columns:
    security = security.filter(pl.col('ticker') == ticker.upper())
display(security)

## Optional local dashboard

From the repository root, launch the read-only Streamlit app for the newest completed expanded report:

```powershell
.venv/Scripts/python -m trading_pipeline.dashboard
```

Use `--expanded-report "$env:TRADING_REPORT_DIR"` to pin a particular expanded report, or `--report` for a Slice 1 report. To launch from this notebook, set `LAUNCH_TRADING_DASHBOARD=1`; the app detects the report type and does not train or modify source artefacts. The inline tables and plots above remain the evidence view when Streamlit is unavailable.

In [ ]:
# Set LAUNCH_TRADING_DASHBOARD=1 for an interactive local session.
# The default remains non-blocking so clean notebook execution can finish.
if os.environ.get('LAUNCH_TRADING_DASHBOARD') == '1':
    import subprocess, sys
    dashboard = subprocess.Popen([sys.executable, '-m', 'trading_pipeline.dashboard', '--expanded-report' if expanded_report else '--report', str(report_dir)])
    display(Markdown('Dashboard started at the Streamlit URL printed by the subprocess.'))

## Interpretation and student reflection prompts

Generated evidence must be separated from interpretation. Student prompts (answer from your own review; do not copy an AI-authored first-person claim):

- Which observed result changed or challenged your prior understanding, and what table or chart supports that?
- Where did forecast/ranking evidence agree or disagree with after-cost portfolio evidence? What mechanism do you think explains it?
- Which limitations matter most to the claims you can make, and why?
- What implementation detail did you personally verify, how did you verify it, and what remains unclear to you?
- Which AI-generated suggestions did you accept or reject after checking code, tests, or immutable artefacts? Record factual AI actions separately from your own reflection.